# Multiple Linear Regression: Prediksi Rating Film

Notebook ini memakai dataset `movie_rating_dataset_100.csv` untuk mempelajari multiple linear regression. Model memakai fitur `Duration_Minutes`, `Genre`, dan `Votes` untuk memprediksi `Rating` film.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


## 1. Membaca Data CSV

Dataset `movie_rating_dataset_100.csv` berisi data film dengan kolom:

- `Movie_ID`: nomor ID film
- `Title`: judul film
- `Duration_Minutes`: durasi film dalam menit
- `Genre`: genre film
- `Votes`: jumlah voting pengguna
- `Rating`: rating film

In [ ]:
csv_path = Path("../dataset/movie_rating_dataset_100.csv")

if not csv_path.exists():
    csv_path = Path("dataset/movie_rating_dataset_100.csv")

if not csv_path.exists():
    csv_path = Path("../movie_rating_dataset_100(1).csv")

df = pd.read_csv(csv_path)
df.head()


## 2. Memisahkan Fitur dan Target

`X` adalah fitur yang dipakai untuk prediksi. `y` adalah nilai rating yang ingin diprediksi.

In [ ]:
fitur = ["Duration_Minutes", "Genre", "Votes"]
X = df[fitur]
y = df["Rating"]

print("Fitur X:")
display(X.head())

print("Target y:")
display(y.head())


## 3. Melatih Model Linear Regression

In [ ]:
def buat_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


preprocessor = ColumnTransformer(
    transformers=[
        ("genre", buat_one_hot_encoder(), ["Genre"]),
    ],
    remainder="passthrough",
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression()),
    ]
)

model.fit(X, y)

print("Model berhasil dilatih.")
print(f"Intercept: {model.named_steps['regressor'].intercept_:.2f}")


## 4. Memprediksi Rating Film Baru

In [ ]:
film_baru = pd.DataFrame(
    [[140, "Drama", 250000]],
    columns=fitur,
)

prediksi_rating = model.predict(film_baru)[0]

print("Data film baru:")
display(film_baru)
print(f"Prediksi rating film baru: {prediksi_rating:.2f}")


## 5. Visualisasi dengan Matplotlib

Visualisasi dibuat dengan membandingkan rating asli dan rating prediksi. Jika titik dekat dengan garis merah, prediksi model semakin dekat dengan rating asli.

In [ ]:
rating_prediksi = model.predict(X)

plt.scatter(y, rating_prediksi, color="blue", alpha=0.6, label="Data film")
plt.plot(
    [y.min(), y.max()],
    [y.min(), y.max()],
    color="red",
    label="Prediksi ideal",
)

plt.title("Rating Asli vs Rating Prediksi")
plt.xlabel("Rating asli")
plt.ylabel("Rating prediksi")
plt.legend()
plt.grid(alpha=0.3)
plt.show()
